# Steric clash ratios — FKBP15 and ENTR1 vs. the WASH core

Two independent checks of whether an accessory subunit's predicted pose is
sterically compatible with the WASH core (WASHC1/2C/3/4/5, chains A–E):

1. **FKBP15** — `wash_fkbp15.pdb` co-folds FKBP15 (chain G) together with the
   core (chains A–E) in a single prediction, so both are already in the same
   coordinate frame. We directly compute the fraction of FKBP15 atoms/residues
   that clash with the core.
2. **ENTR1** — `washc4_washc5_entr1_1.pdb` only contains WASHC4/WASHC5/ENTR1
   (chains D/E/F), co-folded separately from the full core in `wash_core.pdb`.
   - **Before align**: ENTR1's (chain F) clash ratio against its own co-folded
     partners WASHC4/WASHC5 (D/E), computed directly within
     `washc4_washc5_entr1_1.pdb` (same file, same frame) — i.e. how compatible
     ENTR1 is with the small sub-complex it was actually predicted alongside.
   - **After align**: `washc4_washc5_entr1_1.pdb` is superposed onto
     `wash_core.pdb` via the shared D+E Cα atoms (identical residue numbering
     in both files), and ENTR1's clash ratio is recomputed against the *full*
     WASH core (chains A–E) in its new position — testing whether ENTR1 is
     still compatible once placed alongside the rest of the core
     (WASHC1/WASHC2C/WASHC3), which were absent from the original co-folded
     prediction.

| Chain | Gene    |
|-------|---------|
| A     | WASHC1  |
| B     | WASHC2C |
| C     | WASHC3  |
| D     | WASHC4  |
| E     | WASHC5  |
| F     | ENTR1   |
| G     | FKBP15  |

Data source: `N:\08_NK_structure_prediction\data\WASH_complex\fkbp15_entr1_result_figs`


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl
from Bio.PDB import PDBParser
from Bio.SVDSuperimposer import SVDSuperimposer
from scipy.spatial import cKDTree
import warnings
warnings.filterwarnings('ignore')


In [ ]:
# ── Configuration ────────────────────────────────────────────────────────────
DATA_DIR = Path(r"N:\08_NK_structure_prediction\data\WASH_complex\fkbp15_entr1_result_figs")

FKBP15_COMPLEX_PDB = DATA_DIR / "wash_fkbp15.pdb"             # core (A-E) + FKBP15 (G), co-folded
WASH_CORE_PDB       = DATA_DIR / "wash_core.pdb"                # core only (A-E)
ENTR1_SEP_PDB        = DATA_DIR / "washc4_washc5_entr1_1.pdb"    # WASHC4/WASHC5/ENTR1 (D/E/F), separate prediction

CHAIN_GENE = {"A": "WASHC1", "B": "WASHC2C", "C": "WASHC3",
              "D": "WASHC4", "E": "WASHC5", "F": "ENTR1", "G": "FKBP15"}

CORE_CHAINS = {"A", "B", "C", "D", "E"}
SHARED_CHAINS = {"D", "E"}   # common to WASH_CORE_PDB and ENTR1_SEP_PDB, used for superposition

# Bondi van der Waals radii (Å); only C/N/O/S occur in these AF3 outputs (no hydrogens)
VDW_RADII = {"C": 1.70, "N": 1.55, "O": 1.52, "S": 1.80}
DEFAULT_VDW = 1.70
CLASH_TOLERANCE = 0.4   # Å overlap allowance before calling a pair "clashing" (Word et al. 1999, MolProbity)

OUT_DIR = DATA_DIR / "clash_metrics"
OUT_DIR.mkdir(parents=True, exist_ok=True)

FIGURE_DIR = (
    Path(r"N:\08_NK_structure_prediction\XL_MOPLC\XL_complex_structure\result")
    / "fkbp15_entr1_clash_metrics"
)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

print(f"CSV output    → {OUT_DIR}")
print(f"Figure output → {FIGURE_DIR}")


## Clash definition

Two heavy atoms from **different chains** are called *clashing* if their
distance is below the sum of their van der Waals radii minus a 0.4 Å overlap
allowance (the standard MolProbity "bad clash" threshold — small enough that
legitimate polar contacts, e.g. an N···O hydrogen bond at ~2.9 Å, are not
flagged). An atom is a *clash atom* if it clashes with at least one atom of
the partner chain set; the *atom-level clash ratio* is the fraction of atoms
in the subunit of interest that are clash atoms. Since we only ever compare
atoms across separate chains, no covalent-bond exclusion list is needed.

Part 3 below additionally reports a *residue-level* clash ratio: a residue is
a "clash residue" if **any** of its heavy atoms is a clash atom, and the
ratio is the fraction of the subunit's residues that are clash residues. This
is coarser than the atom-level ratio (one bad atom flags the whole residue)
and is often closer to what a modeller means by "how much of the subunit is
clashing".


In [ ]:
# ── Helper functions ──────────────────────────────────────────────────────────
_pdb_parser = PDBParser(QUIET=True)


def load_atoms(pdb_path, chains):
    # Return (coords[N,3], elements[N]) for all heavy atoms in the given chain set.
    structure = _pdb_parser.get_structure("s", str(pdb_path))
    coords, elements = [], []
    for model in structure:
        for chain in model:
            if chain.id not in chains:
                continue
            for res in chain:
                if res.id[0] != " ":
                    continue
                for atom in res:
                    coords.append(atom.get_coord())
                    elements.append((atom.element or atom.get_name()[0]).strip())
        break  # only first model
    return np.array(coords, dtype=float), np.array(elements)


def load_atoms_with_resid(pdb_path, chains):
    # Return (coords[N,3], elements[N], res_ids[N]) for all heavy atoms in the given chain set.
    # res_ids identifies each atom's residue as "<chain_id><res_id>" (unique within `chains`).
    structure = _pdb_parser.get_structure("s", str(pdb_path))
    coords, elements, res_ids = [], [], []
    for model in structure:
        for chain in model:
            if chain.id not in chains:
                continue
            for res in chain:
                if res.id[0] != " ":
                    continue
                for atom in res:
                    coords.append(atom.get_coord())
                    elements.append((atom.element or atom.get_name()[0]).strip())
                    res_ids.append(f"{chain.id}{res.id[1]}")
        break
    return np.array(coords, dtype=float), np.array(elements), np.array(res_ids)


def atom_clash_mask(coords_a, elements_a, coords_b, elements_b, tolerance=CLASH_TOLERANCE):
    # Boolean mask over atoms in set A: True where the atom clashes (VDW-overlap > tolerance)
    # with at least one atom in set B.
    radii_b = np.array([VDW_RADII.get(e, DEFAULT_VDW) for e in elements_b])
    query_radius = 2 * max(VDW_RADII.values())  # generous upper bound (S+S), filtered exactly below
    tree_b = cKDTree(coords_b)
    neighbor_lists = tree_b.query_ball_point(coords_a, r=query_radius)

    mask = np.zeros(len(coords_a), dtype=bool)
    for i, neighbors in enumerate(neighbor_lists):
        if not neighbors:
            continue
        r_a = VDW_RADII.get(elements_a[i], DEFAULT_VDW)
        d = np.linalg.norm(coords_b[neighbors] - coords_a[i], axis=1)
        threshold = r_a + radii_b[neighbors] - tolerance
        mask[i] = np.any(d < threshold)
    return mask


def clash_ratio(coords_a, elements_a, coords_b, elements_b, tolerance=CLASH_TOLERANCE):
    # Atom-level clash ratio: fraction of atoms in set A that clash with set B.
    mask = atom_clash_mask(coords_a, elements_a, coords_b, elements_b, tolerance)
    n_clash, n_total = int(mask.sum()), len(mask)
    return n_clash / n_total, n_clash, n_total


def residue_clash_ratio(clash_mask, res_ids):
    # Residue-level clash ratio: fraction of unique residues (in `res_ids`) with
    # at least one clashing atom, given a precomputed per-atom `clash_mask`.
    per_residue = pd.Series(clash_mask).groupby(np.asarray(res_ids)).any()
    n_clash_res, n_res = int(per_residue.sum()), len(per_residue)
    return n_clash_res / n_res, n_clash_res, n_res


def ca_coords_by_resid(pdb_path, chains):
    # Return {(chain_id, res_id): CA coord} for the given chain set.
    structure = _pdb_parser.get_structure("s", str(pdb_path))
    coords = {}
    for model in structure:
        for chain in model:
            if chain.id not in chains:
                continue
            for res in chain:
                if res.id[0] != " " or "CA" not in res:
                    continue
                coords[(chain.id, res.id[1])] = res["CA"].get_coord()
        break
    return coords


def superpose(mobile_pdb, fixed_pdb, chains):
    # Fit `mobile_pdb`'s CA atoms (given chain set) onto `fixed_pdb`'s; return (rot, tran, rmsd, n).
    mobile = ca_coords_by_resid(mobile_pdb, chains)
    fixed  = ca_coords_by_resid(fixed_pdb, chains)
    keys = sorted(set(mobile) & set(fixed))
    mov = np.array([mobile[k] for k in keys])
    fix = np.array([fixed[k] for k in keys])

    svd = SVDSuperimposer()
    svd.set(fix, mov)   # reference=fixed, moving=mov -> rot/tran maps mov onto fix
    svd.run()
    rot, tran = svd.get_rotran()
    return rot, tran, svd.get_rms(), len(keys)


## Part 1 — FKBP15 clash-atom ratio (`wash_fkbp15.pdb`)


In [ ]:
# ── FKBP15 (chain G) vs. WASH core (chains A-E), same coordinate frame ────────
coords_fkbp15, el_fkbp15, res_fkbp15 = load_atoms_with_resid(FKBP15_COMPLEX_PDB, chains={"G"})
coords_core_wf, el_core_wf = load_atoms(FKBP15_COMPLEX_PDB, chains=CORE_CHAINS)

fkbp15_ratio, fkbp15_n_clash, fkbp15_n_tot = clash_ratio(
    coords_fkbp15, el_fkbp15, coords_core_wf, el_core_wf
)

print(f"FKBP15 atoms: {fkbp15_n_tot}   WASH-core atoms: {len(coords_core_wf)}")
print(f"FKBP15 clash-atom ratio = {fkbp15_ratio:.2%}  ({fkbp15_n_clash}/{fkbp15_n_tot} atoms)")


## Part 2 — ENTR1 clash-atom ratio, before vs. after aligning to `wash_core.pdb`

`washc4_washc5_entr1_1.pdb` (chains D/E/F) is a separate prediction from
`wash_core.pdb` (chains A–E). **Before** align, ENTR1's clash ratio is
computed against its own co-folded WASHC4/WASHC5 (D/E), directly within
`washc4_washc5_entr1_1.pdb` — same file, same frame. **After** align, the
separate model is superposed onto `wash_core.pdb` using the shared D+E Cα
atoms (identical residue numbering in both files), and ENTR1's clash ratio is
recomputed against the *full* core (chains A–E) in that new frame.


In [ ]:
# ── Load ENTR1 (chain F), its co-folded D/E, and the full wash_core ───────────
coords_entr1, el_entr1, res_entr1 = load_atoms_with_resid(ENTR1_SEP_PDB, chains={"F"})
coords_de_sep, el_de_sep = load_atoms(ENTR1_SEP_PDB, chains=SHARED_CHAINS)
coords_wash_core, el_wash_core = load_atoms(WASH_CORE_PDB, chains=CORE_CHAINS)

print(f"ENTR1 atoms: {len(coords_entr1)}   co-folded D+E atoms: {len(coords_de_sep)}"
      f"   wash_core (A-E) atoms: {len(coords_wash_core)}")

# Before alignment: ENTR1 vs. its own co-folded WASHC4/WASHC5 (same file, same frame)
entr1_ratio_before, entr1_n_clash_before, entr1_n_tot = clash_ratio(
    coords_entr1, el_entr1, coords_de_sep, el_de_sep
)
print(f"[before align] ENTR1 vs co-folded D+E clash-atom ratio = {entr1_ratio_before:.2%}"
      f"  ({entr1_n_clash_before}/{entr1_n_tot} atoms)")

# Superpose washc4_washc5_entr1_1.pdb onto wash_core.pdb via shared D+E Ca atoms
rot, tran, rmsd_de, n_de = superpose(ENTR1_SEP_PDB, WASH_CORE_PDB, SHARED_CHAINS)
_, _, rmsd_d, n_d = superpose(ENTR1_SEP_PDB, WASH_CORE_PDB, {"D"})
_, _, rmsd_e, n_e = superpose(ENTR1_SEP_PDB, WASH_CORE_PDB, {"E"})
print(f"\nSuperposition RMSD (D+E Ca, n={n_de}): {rmsd_de:.2f} A")
print(f"  chain D alone (n={n_d}): {rmsd_d:.2f} A")
print(f"  chain E alone (n={n_e}): {rmsd_e:.2f} A")

coords_entr1_aligned = coords_entr1 @ rot + tran

# After alignment: ENTR1 (aligned) vs. the full wash_core (A-E)
entr1_ratio_after, entr1_n_clash_after, _ = clash_ratio(
    coords_entr1_aligned, el_entr1, coords_wash_core, el_wash_core
)
print(f"\n[after align]  ENTR1 vs full wash_core clash-atom ratio = {entr1_ratio_after:.2%}"
      f"  ({entr1_n_clash_after}/{entr1_n_tot} atoms)")


## Part 3 — Residue-level clash ratios

Atom-level ratios can look small even when a meaningful chunk of the subunit
is implicated, since a residue with one clashing atom is just as "bad" as one
with ten. Here we reuse the same atom-level clash calls from Parts 1–2 and
regroup them by residue: a residue counts as clashing if **any** of its
atoms does.


In [ ]:
# ── Residue-level clash ratios (reuse atom-level clash masks from Parts 1-2) ──
mask_fkbp15       = atom_clash_mask(coords_fkbp15, el_fkbp15, coords_core_wf, el_core_wf)
mask_entr1_before = atom_clash_mask(coords_entr1, el_entr1, coords_de_sep, el_de_sep)
mask_entr1_after  = atom_clash_mask(coords_entr1_aligned, el_entr1, coords_wash_core, el_wash_core)

fkbp15_res_ratio, fkbp15_res_n_clash, fkbp15_res_n_tot = residue_clash_ratio(mask_fkbp15, res_fkbp15)
entr1_res_ratio_before, entr1_res_n_clash_before, entr1_res_n_tot = residue_clash_ratio(mask_entr1_before, res_entr1)
entr1_res_ratio_after, entr1_res_n_clash_after, _ = residue_clash_ratio(mask_entr1_after, res_entr1)

print(f"FKBP15 residue-level clash ratio                = {fkbp15_res_ratio:.2%}"
      f"  ({fkbp15_res_n_clash}/{fkbp15_res_n_tot} residues)")
print(f"ENTR1 [before align, vs co-folded D+E] residues = {entr1_res_ratio_before:.2%}"
      f"  ({entr1_res_n_clash_before}/{entr1_res_n_tot} residues)")
print(f"ENTR1 [after align, vs full wash_core] residues = {entr1_res_ratio_after:.2%}"
      f"  ({entr1_res_n_clash_after}/{entr1_res_n_tot} residues)")


## Summary


In [ ]:
# ── Summary table + export ─────────────────────────────────────────────────────
summary_df = pd.DataFrame([
    {"subunit": "FKBP15", "condition": "co-folded with core",
     "n_clash_atoms": fkbp15_n_clash, "n_total_atoms": fkbp15_n_tot, "atom_clash_ratio": round(fkbp15_ratio, 4),
     "n_clash_residues": fkbp15_res_n_clash, "n_total_residues": fkbp15_res_n_tot, "residue_clash_ratio": round(fkbp15_res_ratio, 4)},
    {"subunit": "ENTR1", "condition": "before align (vs co-folded D+E)",
     "n_clash_atoms": entr1_n_clash_before, "n_total_atoms": entr1_n_tot, "atom_clash_ratio": round(entr1_ratio_before, 4),
     "n_clash_residues": entr1_res_n_clash_before, "n_total_residues": entr1_res_n_tot, "residue_clash_ratio": round(entr1_res_ratio_before, 4)},
    {"subunit": "ENTR1", "condition": "after align (vs full wash_core)",
     "n_clash_atoms": entr1_n_clash_after, "n_total_atoms": entr1_n_tot, "atom_clash_ratio": round(entr1_ratio_after, 4),
     "n_clash_residues": entr1_res_n_clash_after, "n_total_residues": entr1_res_n_tot, "residue_clash_ratio": round(entr1_res_ratio_after, 4)},
])
summary_csv = OUT_DIR / "clash_ratios_fkbp15_entr1.csv"
summary_df.to_csv(summary_csv, index=False)
print(f"Saved → {summary_csv}\n")
summary_df


In [ ]:
# ── Bar chart: atom-level vs residue-level clash ratios ───────────────────────
mpl.rcParams.update({
    "font.family": "sans-serif", "font.sans-serif": ["Arial", "Helvetica", "DejaVu Sans"],
    "font.size": 7, "axes.titlesize": 8, "axes.labelsize": 7,
    "xtick.labelsize": 6, "ytick.labelsize": 6, "legend.fontsize": 6, "legend.frameon": False,
    "axes.linewidth": 0.5, "axes.spines.top": False, "axes.spines.right": False,
    "figure.dpi": 150, "savefig.dpi": 300, "savefig.bbox": "tight",
    "pdf.fonttype": 42, "ps.fonttype": 42,
})
NC1 = 89 / 25.4  # single column ~= 3.50"

labels = ["FKBP15\n(co-folded)", "ENTR1\n(before align)", "ENTR1\n(after align)"]
atom_vals = summary_df["atom_clash_ratio"].values
res_vals  = summary_df["residue_clash_ratio"].values
x = np.arange(len(labels))
width = 0.35

fig, ax = plt.subplots(figsize=(NC1 * 1.3, NC1 * 0.9), constrained_layout=True)
ax.bar(x - width / 2, atom_vals, width=width, label="atom-level", color="#0072B2")
ax.bar(x + width / 2, res_vals, width=width, label="residue-level", color="#D55E00")
for xi, v in zip(x - width / 2, atom_vals):
    ax.text(xi, v, f"{v:.1%}", ha="center", va="bottom", fontsize=5.5)
for xi, v in zip(x + width / 2, res_vals):
    ax.text(xi, v, f"{v:.1%}", ha="center", va="bottom", fontsize=5.5)
ax.set_xticks(x)
ax.set_xticklabels(labels)
ax.set_ylabel("Clash ratio")
ax.set_ylim(0, max(atom_vals.max(), res_vals.max()) * 1.3)
ax.set_title("Steric clash ratios vs. WASH core", fontsize=8)
ax.legend(loc="upper left")

fig_path = FIGURE_DIR / "clash_ratios_fkbp15_entr1.svg"
fig.savefig(fig_path, format="svg")
print(f"Saved → {fig_path}")
plt.show()
